# Extensión del semillerío del 9106 a k=50

Agrega 30 semillas nuevas al semillerío del 9106 (reusa las 20 que ya están en `./semillas/`). Las 30 semillas nuevas son EXACTAMENTE las mismas que usamos para el 9105 k=50, así el ensemble multi-modelo final va a tener 50 pares alineados por semilla.

**Requisito:** el pipeline del 9106 debe estar cargado en memoria (`dfinal_train`, `param_final`, `campos_buenos`, `mfuture`, `dfuture`). Si perdiste esa sesión, correr primero el 9106 hasta que arme `param_final`.

In [ ]:
# --- Sanity check: objetos del 9106 en memoria ---
require("data.table")
require("lightgbm")

obligatorios <- c("dfinal_train", "param_final", "campos_buenos", "mfuture", "dfuture")
faltantes <- obligatorios[!sapply(obligatorios, exists)]

if (length(faltantes) > 0) {
  stop("Faltan objetos en memoria: ", paste(faltantes, collapse = ", "),
       "\nCorré el notebook 9106 hasta la celda que arma param_final antes de continuar.")
}

cat("Todos los objetos necesarios están en memoria. OK para continuar.\n")
cat("num_iterations final: ", param_final$num_iterations, "\n")
cat("num_leaves final:     ", param_final$num_leaves, "\n")
cat("learning_rate final:  ", param_final$learning_rate, "\n")
cat("(esperados 9106: niter=379, leaves=384, lr=0.01394)\n")

In [ ]:
# --- Definición de las 50 semillas ---

PARAM$semillerio$semillas_k20 <- c(
  804043, 653561, 703903, 439693, 665857,
  246319, 719179, 688511, 678859, 759179,
  748567, 319687, 771091, 684007, 514853,
  377749, 329977, 757927, 724837, 216973
)

# 30 semillas nuevas (mismas que 9105 k=50 para alinear los pares del ensemble)
PARAM$semillerio$semillas_nuevas30 <- c(
  287333, 656119, 694919, 189817, 867463,
  791801, 804317, 680831, 330917, 595951,
  777571, 662339, 202667, 526159, 509389,
  865993, 749471, 398833, 153269, 969637,
  374683, 678481, 799333, 687083, 941207,
  213349, 735659, 872843, 803207, 414913
)

PARAM$semillerio$semillas_k50 <- c(
  PARAM$semillerio$semillas_k20,
  PARAM$semillerio$semillas_nuevas30
)

stopifnot(length(PARAM$semillerio$semillas_k50) == 50)
stopifnot(length(unique(PARAM$semillerio$semillas_k50)) == 50)

dir.create("semillas", showWarnings = FALSE)
ya_entrenadas <- sapply(PARAM$semillerio$semillas_k50, function(s) {
  file.exists(paste0("semillas/prediccion_semilla_", s, ".txt"))
})

cat("Ya en disco: ", sum(ya_entrenadas), "\n")
cat("A entrenar:  ", sum(!ya_entrenadas), "\n")
cat("(esperado: 20 en disco, 30 a entrenar)\n")

In [ ]:
# --- Loop de entrenamiento de las 30 semillas nuevas ---
# Estimado: 30 semillas × ~1.5-2 min c/u = 45-60 min

semillas_pendientes <- PARAM$semillerio$semillas_k50[!ya_entrenadas]

for (i in seq_along(semillas_pendientes)) {

  semilla <- semillas_pendientes[i]
  cat(format(Sys.time(), "%X"),
      " - Nueva semilla ", i, "/", length(semillas_pendientes),
      " = ", semilla, "\n", sep = "")

  param_semilla <- param_final
  param_semilla$seed <- semilla

  modelo_i <- lgb.train(
    data = dfinal_train,
    param = param_semilla,
    verbose = -100
  )

  prob_i <- predict(modelo_i, mfuture)

  tb_pred_i <- dfuture[, list(numero_de_cliente)]
  tb_pred_i[, prob := prob_i]
  fwrite(tb_pred_i,
    file = paste0("semillas/prediccion_semilla_", semilla, ".txt"),
    sep = "\t"
  )

  rm(modelo_i, prob_i, tb_pred_i)
  gc(full = TRUE, verbose = FALSE)
}

cat("\nEntrenamiento completado. Las 50 semillas del 9106 están en disco.\n")